# Database Lens — OLTP and OLAP

A read-only window onto both databases.

* **OLTP** is RDS PostgreSQL, schema `aimternet_oltp` — current operational state: who is
  checked in, what stock is left, the points ledger.
* **OLAP** is Redshift, schema `aimternet_olap` — the analytical warehouse: facts and
  dimensions built from the Gold layer.

Safety is structural, not a promise. This notebook connects through the `aimternet_ro` role,
which holds `SELECT` and nothing else, inside a transaction pinned `READ ONLY`. An `UPDATE`
typed into a cell below fails twice over.

> Both instances are shared with unrelated coursework. Every query here is scoped to our own
> schemas.

In [1]:
import pandas as pd
from aimternet.config.settings import settings
from aimternet.observability import db_lens

pd.set_option("display.width", 300)
pd.set_option("display.max_columns", 40)

cfg = settings()
print(f"OLTP : {cfg.pg_host}  schema={cfg.pg_schema}  as={cfg.pg_ro_user} (read-only)")
print(f"OLAP : {cfg.redshift_host}  schema={cfg.redshift_schema}")

OLTP : sampledb0.c6rcg6kqa73f.us-east-1.rds.amazonaws.com  schema=aimternet_oltp  as=aimternet_ro (read-only)
OLAP : deds2026.cbvkrohrknsq.us-east-1.redshift.amazonaws.com  schema=aimternet_olap


## 1. What is in the operational database?

In [2]:
db_lens.oltp_tables()

,table_name,estimated_rows,total_size
0,api_idempotency,-1,80 kB
1,concession_items,10,32 kB
2,concession_order_items,29672,10 MB
3,concession_purchases,21077,8592 kB
4,load_checkpoint,1488,776 kB
5,load_manifest,1864,2560 kB
6,member_points_ledger,55514,24 MB
7,members,1200,784 kB
8,pipeline_watermark,-1,32 kB
9,quarantine_records,-1,32 kB


In [3]:
# Exact counts for the business tables. Compare these against the reconciliation report.
db_lens.oltp_exact_counts()

,table_name,rows
0,concession_items,10
1,concession_order_items,29692
2,concession_purchases,21093
3,member_points_ledger,55602
4,members,1200
5,rental_transactions,28358
6,workstations,175


## 2. The floor right now

Open rentals are rows with no `session_end_utc`. The partial unique indexes
`one_open_rental_per_workstation` and `one_open_rental_per_member` make it impossible for a
workstation or a member to appear here twice.

In [4]:
open_now = db_lens.open_rentals()
print(f"{len(open_now)} open rental(s)")
open_now

0 open rental(s)


""


In [5]:
db_lens.workstation_status()

,zone_classification,status,workstations
0,Standard Zone,AVAILABLE,100
1,Streamer Pods,AVAILABLE,25
2,VIP Esports Zone,AVAILABLE,50


## 3. Where the money came from

In [7]:
revenue = db_lens.revenue_by_zone(limit_days=5)
revenue

,zone_classification,day,rentals,net_revenue
0,Standard Zone,2026-09-03,70,6555.00
1,Streamer Pods,2026-09-03,1,360.00
2,Standard Zone,2026-08-31,231,26950.00
3,Streamer Pods,2026-08-31,60,18416.00
4,VIP Esports Zone,2026-08-31,127,25264.00
5,Standard Zone,2026-08-30,260,29250.00
6,Streamer Pods,2026-08-30,60,15568.00
7,VIP Esports Zone,2026-08-30,118,21388.00
8,Standard Zone,2026-08-29,257,28565.00
9,Streamer Pods,2026-08-29,68,19172.00


In [8]:
# Same data as a chart, when the table gets long enough to stop being readable.
if not revenue.empty:
    pivot = revenue.pivot_table(
        index="day", columns="zone_classification", values="net_revenue", aggfunc="sum"
    ).fillna(0)
    ax = pivot.plot(kind="bar", stacked=True, figsize=(11, 4))
    ax.set_ylabel("net revenue (PHP)")
    ax.set_title("Rental revenue by zone")
else:
    print("No rentals loaded yet — run `make bootstrap` first.")

TypeError: no numeric data to plot

## 4. Data-quality spot checks

Two checks worth running by hand whenever something looks off.

**The D2 cohort.** 840 members (`M-1001`–`M-1840`) are referenced by transactions but appear
in no source file. Under the `synthesize_stub` policy they are inserted as flagged stubs, so
they should be visible here rather than hidden.

In [9]:
db_lens.backfilled_members()

,source_system,is_backfilled,members,first_id,last_id
0,DERIVED_FROM_TRANSACTIONS,True,840,M-1001,M-1840
1,LEGACY_BATCH,False,360,M-1841,M-2200


**Points balance drift.** The ledger is the audit trail; `members.current_points_balance` is a
cache of it. Any row returned here means the cache disagrees with the ledger — the ledger wins.
An empty result is the healthy answer.

In [10]:
drift = db_lens.points_balance_drift()
print("clean — every balance matches its ledger" if drift.empty else f"{len(drift)} member(s) drifting")
drift

20 member(s) drifting


,member_id,stored_balance,ledger_sum,drift
0,M-1539,2900,-1770,4670
1,M-1676,3416,-683,4099
2,M-1665,2158,-1879,4037
3,M-1330,3467,-320,3787
4,M-1347,3408,-201,3609
5,M-1398,1933,-1470,3403
6,M-1707,1865,-1470,3335
7,M-1756,2112,-1197,3309
8,M-1277,2741,-467,3208
9,M-1221,1836,-1293,3129


In [11]:
db_lens.top_members(10)

,member_id,first_name,last_name,current_tier,lifetime_spend_amount,current_points_balance,rentals
0,M-1210,Unknown,Member M-1210,Silver,12937.00,636,44
1,M-1176,Unknown,Member M-1176,Standard,11217.00,165,35
2,M-1411,Unknown,Member M-1411,Silver,10917.00,286,39
3,M-1040,Unknown,Member M-1040,Standard,10844.00,46,38
4,M-1420,Unknown,Member M-1420,Standard,10703.00,56,30
5,M-1439,Unknown,Member M-1439,Standard,10672.00,64,37
6,M-1677,Unknown,Member M-1677,Standard,10588.00,30,41
7,M-1201,Unknown,Member M-1201,Standard,10516.00,156,37
8,M-1063,Unknown,Member M-1063,Silver,10483.00,518,36
9,M-1111,Unknown,Member M-1111,Standard,10357.00,76,33


## 5. The analytical warehouse

In [12]:
db_lens.olap_tables()

,table_name
0,agg_workstation_utilization_hourly
1,dim_concession_item
2,dim_date
3,dim_member
4,dim_time
5,dim_workstation
6,fact_concession_line_item
7,fact_concession_sale
8,fact_points_activity
9,fact_rental


In [14]:
# Anything in aimternet_olap is queryable from here. Once Gold is loaded, try:
#   db_lens.olap("SELECT * FROM agg_workstation_utilization_hourly LIMIT 20")
tables = db_lens.olap_tables()
if tables.empty:
    print("Redshift schema is empty — run `make redshift` after `make curate`.")
else:
    display(db_lens.olap(f"SELECT * FROM dim_member LIMIT 10")) #{tables.iloc[0]['table_name']}

,member_key,member_id,first_name,last_name,email,phone_number,tier,valid_from_utc,valid_to_utc,is_current,current_points_balance,lifetime_spend_amount,registered_at_utc,source_system,is_backfilled,run_id,built_at_utc
0,1,M-1001,Unknown,Member M-1001,m-1001@backfilled.invalid,None,Standard,2026-07-05 14:45:00+00:00,2026-07-25 12:00:00+00:00,False,2,7115.00,2026-07-05 14:45:00+00:00,DERIVED_FROM_TRANSACTIONS,True,curate-0c94dca8d657,2026-09-03 14:48:14.964483+00:00
1,2,M-1001,Unknown,Member M-1001,m-1001@backfilled.invalid,None,Silver,2026-07-25 12:00:00+00:00,NaT,True,2,7115.00,2026-07-05 14:45:00+00:00,DERIVED_FROM_TRANSACTIONS,True,curate-0c94dca8d657,2026-09-03 14:48:14.964483+00:00
2,3,M-1002,Unknown,Member M-1002,m-1002@backfilled.invalid,None,Silver,2026-07-01 11:10:00+00:00,2026-07-20 13:00:00+00:00,False,552,6104.00,2026-07-01 11:10:00+00:00,DERIVED_FROM_TRANSACTIONS,True,curate-0c94dca8d657,2026-09-03 14:48:14.964483+00:00
3,4,M-1002,Unknown,Member M-1002,m-1002@backfilled.invalid,None,Gold,2026-07-20 13:00:00+00:00,NaT,True,552,6104.00,2026-07-01 11:10:00+00:00,DERIVED_FROM_TRANSACTIONS,True,curate-0c94dca8d657,2026-09-03 14:48:14.964483+00:00
4,5,M-1003,Unknown,Member M-1003,m-1003@backfilled.invalid,None,Standard,2026-07-03 12:00:00+00:00,2026-07-10 01:30:00+00:00,False,117,6088.00,2026-07-03 12:00:00+00:00,DERIVED_FROM_TRANSACTIONS,True,curate-0c94dca8d657,2026-09-03 14:48:14.964483+00:00
5,6,M-1003,Unknown,Member M-1003,m-1003@backfilled.invalid,None,Silver,2026-07-10 01:30:00+00:00,NaT,True,117,6088.00,2026-07-03 12:00:00+00:00,DERIVED_FROM_TRANSACTIONS,True,curate-0c94dca8d657,2026-09-03 14:48:14.964483+00:00
6,7,M-1004,Unknown,Member M-1004,m-1004@backfilled.invalid,None,Standard,2026-07-01 10:45:00+00:00,2026-07-01 19:05:00+00:00,False,279,7864.00,2026-07-01 10:45:00+00:00,DERIVED_FROM_TRANSACTIONS,True,curate-0c94dca8d657,2026-09-03 14:48:14.964483+00:00
7,8,M-1004,Unknown,Member M-1004,m-1004@backfilled.invalid,None,Silver,2026-07-01 19:05:00+00:00,NaT,True,279,7864.00,2026-07-01 10:45:00+00:00,DERIVED_FROM_TRANSACTIONS,True,curate-0c94dca8d657,2026-09-03 14:48:14.964483+00:00
8,9,M-1005,Unknown,Member M-1005,m-1005@backfilled.invalid,None,Standard,2026-07-03 14:15:00+00:00,2026-07-22 05:25:00+00:00,False,26,7777.00,2026-07-03 14:15:00+00:00,DERIVED_FROM_TRANSACTIONS,True,curate-0c94dca8d657,2026-09-03 14:48:14.964483+00:00
9,10,M-1005,Unknown,Member M-1005,m-1005@backfilled.invalid,None,Silver,2026-07-22 05:25:00+00:00,NaT,True,26,7777.00,2026-07-03 14:15:00+00:00,DERIVED_FROM_TRANSACTIONS,True,curate-0c94dca8d657,2026-09-03 14:48:14.964483+00:00


## 6. Your own query

`db_lens.oltp(sql)` and `db_lens.olap(sql)` both return a DataFrame. The OLTP one is
read-only; the OLAP one runs as the cluster user, so keep it to `SELECT`.

In [15]:
db_lens.oltp('''
    SELECT current_tier, count(*) AS members, sum(lifetime_spend_amount) AS lifetime_spend
    FROM members
    GROUP BY current_tier
    ORDER BY lifetime_spend DESC NULLS LAST
''')

,current_tier,members,lifetime_spend
0,Standard,956,4471868.00
1,Silver,170,1185977.00
2,Gold,74,473183.00
